In [1]:
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import random_split
from model.T_learner import TLearner
from engine.trainer import Trainer
from data.dataset import load_ihdp_data, IPDHDataset, IPHDDataLoader
from utils.metric import pehe, policy_risk, uplift_curve, uplift_auc_score, qini_curve_industry, qini_auc_score_industry
from utils.plot import plot_uplift_curve, plot_qini_curve

In [2]:
train_path = Path('data/ihdp_npci_1-100.train.npz')
data_dict = load_ihdp_data(train_path)
full_dataset = IPDHDataset(data_dict)
train_ratio = 0.8
train_size = int(train_ratio * len(full_dataset))
valid_size = len(full_dataset) - train_size
train_set, valid_set = random_split(full_dataset, [train_size, valid_size])
train_loader = IPHDDataLoader(train_set, batch_size=32, shuffle=True)
valid_loader = IPHDDataLoader(valid_set, batch_size=32, shuffle=False)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_t = TLearner(x_dim = 25, hidden_dim = 64, type = 'treatment').to(device)
optimizer_t = torch.optim.Adam(model_t.parameters(), lr=1e-3)
trainer_t = Trainer(model_t, optimizer_t, device)
trainer_t.run(train_loader, valid_loader, epochs=100)


model_c = TLearner(x_dim = 25, hidden_dim = 64, type = 'control').to(device)
optimizer_c = torch.optim.Adam(model_c.parameters(), lr=1e-3)
trainer_c = Trainer(model_c, optimizer_c, device)
trainer_c.run(train_loader, valid_loader, epochs=100)

Ep  1 | train=39.4762 | val=34.6284
Ep  2 | train=25.8862 | val=11.6844
Ep  3 | train=4.8820 | val=1.2763
Ep  4 | train=2.3292 | val=1.5342
Ep  5 | train=1.6908 | val=0.9676
Ep  6 | train=1.6718 | val=0.9494
Ep  7 | train=1.7551 | val=0.8907
Ep  8 | train=1.4037 | val=0.9110
Ep  9 | train=1.4257 | val=0.9248
Ep 10 | train=1.4189 | val=0.8660
Ep 11 | train=1.2859 | val=0.8741
Ep 12 | train=1.2069 | val=0.9202
Ep 13 | train=1.0954 | val=1.0374
Ep 14 | train=1.0673 | val=0.9434
Ep 15 | train=1.0773 | val=0.9429
✅ 早停触发！停止训练
Ep  1 | train=6.8483 | val=5.8962
Ep  2 | train=3.0596 | val=2.2051
Ep  3 | train=1.9492 | val=1.7962
Ep  4 | train=1.4759 | val=1.4851
Ep  5 | train=1.3074 | val=1.4276
Ep  6 | train=1.2221 | val=1.3453
Ep  7 | train=1.1089 | val=1.2611
Ep  8 | train=1.0862 | val=1.2458
Ep  9 | train=1.0107 | val=1.1802
Ep 10 | train=0.9823 | val=1.1644
Ep 11 | train=0.9400 | val=1.1500
Ep 12 | train=0.8931 | val=1.1713
Ep 13 | train=0.8695 | val=1.1395
Ep 14 | train=0.8384 | val=1.134

In [4]:
test_path = Path('data/ihdp_npci_1-100.test.npz')
test_data = load_ihdp_data(test_path)
test_set = IPDHDataset(test_data)
test_loader = IPHDDataLoader(test_set, batch_size=32, shuffle=False)

In [5]:
model_t.eval()
model_c.eval()
tau_hat_list = []
tau_true_list = []
for batch in test_loader:
    x, t, y, mu0, mu1 = batch
    x = x.to(device)
    # predicted treatment effect
    mu_hat_1 = model_t.predict(x).cpu()
    mu_hat_0 = model_c.predict(x).cpu()
    tau_hat = mu_hat_1 - mu_hat_0
    # true treatment effect
    tau_true = (mu1 - mu0)
    tau_hat_list.append(tau_hat)
    tau_true_list.append(tau_true)

tau_hat = torch.cat(tau_hat_list, dim=0).cpu().numpy()
tau_true = torch.cat(tau_true_list, dim=0).cpu().numpy()

In [6]:
pehe_score = pehe(tau_hat, tau_true)
print("PEHE:", pehe_score)

PEHE: 0.6498613
